# PyTorch Fundamentals: BatchNorm variance: biased vs unbiased

**Solution notebook — Delta Drills #455**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

Verify that nn.BatchNorm2d in train mode uses the BIASED variance. Build the single-channel tensor x = [[[[1,2],[3,4]]],[[[5,6],[7,8]]]] (shape 2,1,2,2) as float32. Create bn = nn.BatchNorm2d(1, affine=False, eps=1e-5), call bn.train(), and run out = bn(x). Independently compute mean and BIASED variance (unbiased=False) over dims (0,2,3) with keepdim=True, then manual = (x-mean)/sqrt(var_biased+eps). Print on line 1 the biased variance as [round(v,4) for v in var_biased.flatten().tolist()]; on line 2 print bool(torch.allclose(out, manual, atol=1e-5)); on line 3 print [round(v,4) for v in out.flatten().tolist()].


<details><summary>💡 Hint (click to reveal)</summary>

Recreate BatchNorm's normalization by hand using the biased per-channel variance with keepdim, then compare against the layer's output with allclose.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch
import torch.nn as nn
x = torch.tensor([[[[1., 2.], [3., 4.]]], [[[5., 6.], [7., 8.]]]])
eps = 1e-5
bn = nn.BatchNorm2d(1, affine=False, eps=eps)
bn.train()
out = bn(x)
mean = x.mean(dim=(0, 2, 3), keepdim=True)
var_biased = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)
manual = (x - mean) / torch.sqrt(var_biased + eps)
print([round(v, 4) for v in var_biased.flatten().tolist()])
print(bool(torch.allclose(out, manual, atol=1e-5)))
print([round(v, 4) for v in out.flatten().tolist()])


## Why this works

In train mode BatchNorm2d normalizes with the biased variance over dims (0,2,3), so the manual formula (x-mean)/sqrt(var_biased+eps) reproduces its output to within tolerance. keepdim=True lets mean and variance broadcast back against x, and allclose confirms the match.
